# 真实零值与自然边界：Two-part、Hurdle、Zero-inflated 与 Fractional response

04 章用 Tobit 模型说明了一类很重要的观测机制：存在一个潜在连续变量 $B_i^*$，但实际观察到的变量不能低于某个边界，于是所有边界外取值都被归并到边界值。例如，潜在净借款需求 $B_i^*$ 可以为负，但实际银行贷款金额 $B_i$ 不能为负，因此 $B_i=\\max(0,B_i^*)$。

本章换一个角度思考同一个企业信贷背景：如果 $B_i=0$ 并不是潜在连续变量被归并到 0，而是企业真实地没有贷款，应该如何建模？更进一步，如果结果变量是贷款笔数、贷款结构比例或绿色贷款占比，又该如何处理自然边界和大量零值？

本章的主线是：模型选择不应从“$y$ 有没有很多 0”开始，而应从“0 是如何生成的”开始。

## 1. 模型地图：先判断 0 与边界的含义

在受限因变量模型中，最容易犯的错误是：只要看到很多 0，就直接想到 Tobit。实际上，同样是大量 0，背后的数据生成机制可以完全不同。

- 如果样本仍在数据中，但边界外的潜在取值被统一记录为 0，这是 **归并 (censoring)**，Tobit 是一个自然的基准模型。
- 如果样本没有进入数据，这是 **截断 (truncation)**，问题已经不是因变量被记录为 0，而是样本选择进入数据的规则不同。
- 如果 0 是真实选择，例如企业确实没有贷款、没有专利、没有出口，则更自然的思路是先建模“是否发生”，再建模“发生多少”。
- 如果结果变量是比例或份额，则因变量天然位于 $[0,1]$，应使用能尊重自然边界的条件均值模型。

![受限因变量模型地图](./figs/limit_dep_alt_fig01_model_map.png){width="95%"}

::: {.callout-tip}
### 本章的核心问题

本章不再把 0 主要理解为“潜在变量被归并到 0”，而是重点讨论真实零值和自然边界。换言之，本章关注的是：企业为什么没有贷款、获得贷款后贷多少、贷款笔数为什么有很多 0，以及比例结果为什么不能简单使用线性模型。
:::

## 2. 为什么贷款金额不一定适合 Tobit？

继续使用企业银行贷款金额的例子。设 $B_i$ 表示企业实际银行贷款金额。在 04 章中，我们可以把 $B_i=0$ 理解为潜在净借款需求 $B_i^*$ 被归并到 0：

$$
B_i=\max(0,B_i^*)
$$

这种设定隐含一个很强的共同机制假设：同一组变量既决定企业是否有正贷款，也决定正贷款金额的大小。

但在很多实际信贷场景中，贷款可能更像两个阶段：

- 第一阶段：企业是否进入银行信贷关系，或者是否获得贷款；
- 第二阶段：在已经获得贷款的企业中，贷款金额是多少。

这两个阶段的决定因素可能不完全相同。例如，`bank_access` 主要影响企业是否能进入信贷关系；但企业一旦获得贷款，实际贷款金额可能更多取决于投资机会和抵押能力。

因此，本章将企业贷款金额写成：

$$
D_i=1(B_i>0)
$$

$$
P(D_i=1\mid z_i)=F(z_i'\gamma)
$$

$$
E(B_i\mid B_i>0,x_i)=m(x_i'\beta)
$$

这就是 Two-part model 的基本思路。

::: {.callout-note}
### Tobit 与 Two-part 的本质差别

Tobit 假定是否出现正值和正值大小来自同一个潜在连续机制；Two-part model 则允许“是否发生”和“发生多少”由两个机制分别决定。因此，Two-part model 不只是“跑两个回归”，而是对数据生成过程作出了不同假设。
:::

## 3. Two-part model：具体的数据生成过程

本章使用一个尽量简洁的企业信贷例子。令 $B_i$ 表示企业银行贷款金额，$D_i=1(B_i>0)$ 表示企业是否获得贷款。

第一部分建模企业是否获得贷款：

$$
P(D_i=1\mid z_i)=F(\gamma_0+\gamma_1 collateral_i+\gamma_2 bank\_access_i)
$$

第二部分建模获得贷款后的贷款金额：

$$
E(B_i\mid B_i>0,x_i)=\exp(\beta_0+\beta_1 collateral_i+\beta_2 opportunity_i)
$$

变量设定如下：

| 变量 | 第一部分 $z_i$ | 第二部分 $x_i$ | 经济含义 |
|---|---:|---:|---|
| `collateral` | 是 | 是 | 抵押能力既影响贷款可得性，也影响获贷后的额度 |
| `bank_access` | 是 | 否 | 银行可得性主要影响企业能否进入信贷关系 |
| `opportunity` | 否 | 是 | 投资机会主要影响企业获贷后愿意借多少 |

这个例子的关键是：`collateral` 是公共变量，`bank_access` 和 `opportunity` 则分别对应概率渠道和强度渠道。

![Two-part model 的概率渠道与强度渠道](./figs/limit_dep_alt_fig02_two_part_mechanism.png){width="90%"}

::: {.callout-important}
### 为什么变量不能机械地放进两个方程？

Two-part model 中的 $z_i$ 和 $x_i$ 不应简单设为完全相同的一组控制变量。变量是否进入第一部分或第二部分，应由经济机制决定。

例如，`bank_access` 代表企业附近银行网点、银企关系或可获得信贷服务的便利程度。它主要影响企业是否能够获得贷款，因此自然进入第一部分。`opportunity` 代表投资机会，企业已经获得贷款后，投资机会越强，所需贷款金额越大，因此自然进入第二部分。

当然，某些变量可以同时进入两个方程。`collateral` 就是这样的变量：抵押能力既能提高获贷概率，也能提高授信额度。
:::

## 4. Two-part model 的非条件期望

Two-part model 的核心不是两张回归表，而是非条件期望的乘法结构。令：

$$
p_i=P(D_i=1\mid z_i)
$$

$$
\mu_i^+=E(B_i\mid B_i>0,x_i)
$$

则企业贷款金额的非条件期望为：

$$
E(B_i\mid z_i,x_i)=p_i\mu_i^+
$$

在本章设定中，如果第一部分使用 Logit，第二部分使用 log link，则：

$$
p_i=F(z_i'\gamma)
$$

$$
\mu_i^+=\exp(x_i'\beta)
$$

因此：

$$
E(B_i\mid z_i,x_i)=F(z_i'\gamma)\exp(x_i'\beta)
$$

这说明，一个变量对最终贷款金额的影响可能来自两个渠道：改变企业获得贷款的概率，或者改变获得贷款后的贷款金额。

![Tobit 与 Two-part 的机制对比](./figs/limit_dep_alt_fig03_tobit_vs_twopart.png){width="95%"}

## 5. Two-part model 的边际效应

假设某个变量 $w_i$ 同时进入第一部分和第二部分。若第一部分为 $p_i=F(z_i'\gamma)$，第二部分为 $\mu_i^+=\exp(x_i'\beta)$，则：

$$
E(B_i\mid z_i,x_i)=p_i\mu_i^+
$$

对 $w_i$ 求导，有：

$$
\frac{\partial E(B_i\mid z_i,x_i)}{\partial w_i}
=
f(z_i'\gamma)\gamma_w\mu_i^+
+
p_i\mu_i^+\beta_w
$$

其中：

- 第一项 $f(z_i'\gamma)\gamma_w\mu_i^+$ 是 **概率渠道**；
- 第二项 $p_i\mu_i^+\beta_w$ 是 **强度渠道**；
- 两项加总才是变量对非条件贷款金额的总影响。

如果变量只进入第一部分，例如 `bank_access`，则：

$$
\frac{\partial E(B_i\mid z_i,x_i)}{\partial bank\_access_i}
=
f(z_i'\gamma)\gamma_{bank}\mu_i^+
$$

如果变量只进入第二部分，例如 `opportunity`，则：

$$
\frac{\partial E(B_i\mid z_i,x_i)}{\partial opportunity_i}
=
p_i\mu_i^+\beta_{opp}
$$

平均边际效应为：

$$
AME_w=
\frac{1}{n}\sum_{i=1}^n
\left[
 f(z_i'\gamma)\gamma_w\mu_i^+
 +p_i\mu_i^+\beta_w
\right]
$$

::: {.callout-tip}
### 结果解释的核心

Two-part model 的结果不能只说“第一部分显著”或“第二部分显著”。更完整的解释应当回答：这个变量是通过提高发生概率影响 $B_i$，还是通过提高正值条件均值影响 $B_i$，或者二者同时存在？
:::

In [1]:
# ============================================================
# 读取 Two-part model 的核心结果
# ============================================================
import pandas as pd

coef_table = pd.read_csv("./data/credit_two_part_coef.csv")
ame_table = pd.read_csv("./data/credit_two_part_ame.csv")
summary = pd.read_csv("./data/credit_limit_dep_summary.csv")

summary

,指标,数值
0,样本量,4000.000000
1,贷款金额为 0 的比例,0.499500
2,正贷款金额均值,6.012521
3,贷款笔数为 0 的比例,0.569000
4,绿色贷款占比均值,0.631930


In [2]:
coef_table.round(4)

,模型,解释对象,变量,系数,标准误,t/z 值,p 值
0,第一部分：Logit,是否获得贷款,const,-1.0640,0.0837,-12.7050,0.0
1,第一部分：Logit,是否获得贷款,collateral,2.5591,0.1805,14.1773,0.0
2,第一部分：Logit,是否获得贷款,bank_access,0.7851,0.0390,20.1532,0.0
3,第二部分：log-OLS,正贷款金额,const,1.1742,0.0276,42.5388,0.0
4,第二部分：log-OLS,正贷款金额,collateral,0.7770,0.0543,14.3159,0.0
5,第二部分：log-OLS,正贷款金额,opportunity,0.4845,0.0108,44.9716,0.0


In [3]:
ame_table.round(4)

,变量,概率渠道 AME,强度渠道 AME,总 AME
0,collateral,3.1409,2.3502,5.4911
1,bank_access,0.9637,0.0000,0.9637
2,opportunity,0.0000,1.4653,1.4653


从平均边际效应的角度看：

- `collateral` 同时进入两个方程，因此它的总效应可以拆成概率渠道和强度渠道；
- `bank_access` 只进入第一部分，因此它主要通过提高获贷概率影响非条件贷款金额；
- `opportunity` 只进入第二部分，因此它主要通过提高获贷后的贷款金额影响非条件贷款金额。

![Two-part model 的预测分解](./figs/limit_dep_alt_fig04_twopart_prediction_decomp.png){width="88%"}

## 6. log-OLS、smearing correction 与正值金额模型

在第二部分中，一个常见做法是对正贷款金额取对数后做 OLS：

$$
\log B_i=x_i'\beta+u_i,\quad B_i>0
$$

这样做的好处是能够缓解金额变量右偏分布的问题，也避免预测负贷款金额。但需要注意，从 $\log B_i$ 回到 $B_i$ 时，不能简单写成：

$$
\widehat{E}(B_i\mid B_i>0,x_i)=\exp(x_i'\hat\beta)
$$

因为：

$$
E(B_i\mid B_i>0,x_i)=E\{\exp(x_i'\beta+u_i)\mid x_i\}
$$

若 $u_i$ 不等于 0，直接取指数会低估 level 上的条件均值。一个常见修正是 smearing correction：

$$
\widehat{E}(B_i\mid B_i>0,x_i)
=
\exp(x_i'\hat\beta)\hat S
$$

其中：

$$
\hat S=
\frac{1}{n_+}\sum_{i:B_i>0}\exp(\hat u_i)
$$

::: {.callout-note}
### 为什么要讲 smearing correction？

在金融数据中，金额变量通常右偏严重，对数回归很常见。但教学中如果只说“对数回归后再取指数”，学生会形成错误习惯。smearing correction 提醒我们：模型在 log 尺度上估计，预测目标却可能在 level 尺度上，两者之间不能机械转换。
:::

## 7. Hurdle model：先跨过门槛，再决定正计数

Two-part model 常用于正连续结果；如果结果变量是非负计数，也可以采用类似的“先是否发生，再发生多少”的思路。一个典型例子是企业贷款笔数。

设 $Y_i$ 表示企业一年内的贷款笔数。Hurdle model 的逻辑是：

- 第一阶段决定企业是否跨过 0 这个门槛；
- 第二阶段只在正计数样本中建模 $Y_i=1,2,3,\ldots$。

令 $p_i=P(Y_i>0\mid z_i)$，正计数部分服从零截断计数分布，则：

$$
P(Y_i=0\mid z_i)=1-p_i
$$

$$
P(Y_i=k\mid z_i,x_i)=
p_i\cdot
\frac{f(k\mid \lambda_i)}{1-f(0\mid \lambda_i)},
\quad k=1,2,\ldots
$$

这里的分母 $1-f(0\mid \lambda_i)$ 表示对普通计数分布去掉 0 后重新标准化。

![Hurdle model 的零值与正计数机制](./figs/limit_dep_alt_fig05_hurdle_mechanism.png){width="85%"}

::: {.callout-tip}
### Hurdle 与最低贷款规模

如果某些项目贷款或债券融资存在较高固定成本、审批成本或最低经济规模，那么企业可能先决定是否进入该融资方式；一旦进入，融资规模才成为正值。这类问题的直觉与 Hurdle model 接近。
:::

## 8. Zero-inflated count model：0 有两个来源

Zero-inflated model 也是处理大量 0 的计数模型，但它和 Hurdle model 的零值机制不同。

Zero-inflated model 假定样本中存在两类企业：

- 一类是结构性零值企业，即在机制上不会产生正计数；
- 另一类企业进入普通计数过程，但普通计数过程本身也可能产生 0。

令 $\pi_i$ 表示结构性 0 的概率，$f(k\mid\lambda_i)$ 表示普通计数分布，则：

$$
P(Y_i=0\mid z_i,x_i)=
\pi_i+(1-\pi_i)f(0\mid\lambda_i)
$$

$$
P(Y_i=k\mid z_i,x_i)=
(1-\pi_i)f(k\mid\lambda_i),
\quad k=1,2,\ldots
$$

与 Hurdle model 相比，Zero-inflated model 的关键是：0 有两个来源。

![Zero-inflated model 中的两类零值](./figs/limit_dep_alt_fig06_zero_inflated_sources.png){width="82%"}

| 模型 | 0 的来源 | 正计数部分 |
|---|---|---|
| Hurdle model | 只来自是否跨过门槛 | 零截断计数分布 |
| Zero-inflated model | 结构性 0 + 普通计数过程中的随机 0 | 普通计数分布 |

::: {.callout-important}
### 两者不要机械混用

如果研究问题强调“先是否进入某个活动，再发生多少次”，Hurdle model 的解释更自然。如果研究问题强调“有些企业本质上不会产生该行为，而另一些企业只是当期没有发生”，Zero-inflated model 的解释更自然。
:::

## 9. Fractional response model：比例因变量的自然边界

许多金融变量不是金额，也不是计数，而是比例或份额。例如：

- 银行贷款占总负债比例；
- 绿色贷款占总贷款比例；
- 短期贷款占总贷款比例；
- 某类资产占总资产比例。

这类变量自然位于 $[0,1]$。若直接使用 OLS：

$$
E(s_i\mid x_i)=x_i'\beta
$$

可能出现两个问题：

- 预测值可能小于 0 或大于 1；
- 边际效应被设定为常数，无法反映比例变量接近边界时变化空间变小。

Fractional response model 设定：

$$
E(s_i\mid x_i)=G(x_i'\beta)
$$

其中常用的 $G(\cdot)$ 是 logistic CDF：

$$
G(a)=\frac{\exp(a)}{1+\exp(a)}
$$

此时边际效应为：

$$
\frac{\partial E(s_i\mid x_i)}{\partial x_{ij}}
=
G(x_i'\beta)\{1-G(x_i'\beta)\}\beta_j
$$

平均边际效应为：

$$
AME_j=
\frac{1}{n}\sum_{i=1}^n
G(x_i'\hat\beta)\{1-G(x_i'\hat\beta)\}\hat\beta_j
$$

![Fractional response 与 OLS 预测对比](./figs/limit_dep_alt_fig07_frm_vs_ols.png){width="86%"}

In [4]:
frm_ame = pd.read_csv("./data/credit_frm_ame.csv")
frm_ame.round(4)

,变量,FRM AME,OLS 边际效应
0,collateral,0.3781,0.3771
1,risk,-0.1428,-0.1430


::: {.callout-note}
### FRM 与 Logit 有什么关系？

FRM 中的因变量不是二元变量，而是比例变量。它借用了 Logit 的 S 形条件均值函数，使预测值自然落在 $[0,1]$ 内。估计时通常采用 quasi-MLE，即使比例变量不是真正的 Bernoulli 变量，只要条件均值设定正确，系数估计仍具有良好性质。
:::

## 10. PE 类理财产品：一个适合作为课堂讨论的例子

PE 类理财产品不适合作为 Tobit 的主例，但很适合帮助理解 Two-part / Hurdle 的机制。

如果某类产品有合格投资者要求和最低投资额，例如要求过去三年年均收入不低于某个门槛，或金融资产规模不低于某个门槛，并且最低投资金额为 $M$，那么观测到的投资金额往往具有如下结构：

$$
I_i=
\begin{cases}
0, & D_i=0,\\
M+\tilde I_i, & D_i=1.
\end{cases}
$$

这里的 0 并不是潜在投资额被归并到 0，而是投资者没有参与、没有资格或没有跨过制度门槛。正值部分也不是从接近 0 的地方连续开始，而是从最低投资金额 $M$ 起跳。

因此，这类问题更接近 Two-part 或 Hurdle 的设定，而不是标准 Tobit。

## 11. 本章小结

本章的核心不是把很多模型排成清单，而是帮助读者建立一套判断逻辑：

| 观测现象 | 关键问题 | 更自然的模型 |
|---|---|---|
| 贷款金额有大量 0 | 0 是否是潜在净借款需求被归并？ | Tobit |
| 贷款金额有大量 0 | 是否贷款与贷多少是否是两个机制？ | Two-part model |
| 贷款笔数有大量 0 | 是否先跨过 0，再决定正计数？ | Hurdle model |
| 贷款笔数有大量 0 | 0 是否来自结构性 0 和随机 0 两个来源？ | Zero-inflated model |
| 贷款结构比例位于 $[0,1]$ | 线性预测是否可能越界？ | Fractional response model |

对于实证研究而言，最重要的问题不是“哪一个模型更高级”，而是研究者能否说清楚：数据中的 0、正值、比例边界和缺失值到底来自什么样的经济机制。

## 参考文献

- Cragg, J. G. (1971). Some statistical models for limited dependent variables with application to the demand for durable goods. *Econometrica*, 39(5), 829–844. [Link](https://doi.org/10.2307/1909582), [PDF](http://sci-hub.ren/10.2307/1909582), [Google](https://scholar.google.com/scholar?q=Some+statistical+models+for+limited+dependent+variables+with+application+to+the+demand+for+durable+goods).
- Duan, N. (1983). Smearing estimate: A nonparametric retransformation method. *Journal of the American Statistical Association*, 78(383), 605–610. [Link](https://doi.org/10.1080/01621459.1983.10478017), [PDF](http://sci-hub.ren/10.1080/01621459.1983.10478017), [Google](https://scholar.google.com/scholar?q=Smearing+estimate+A+nonparametric+retransformation+method).
- Mullahy, J. (1986). Specification and testing of some modified count data models. *Journal of Econometrics*, 33(3), 341–365. [Link](https://doi.org/10.1016/0304-4076(86)90002-3), [PDF](http://sci-hub.ren/10.1016/0304-4076(86)90002-3), [Google](https://scholar.google.com/scholar?q=Specification+and+testing+of+some+modified+count+data+models).
- Lambert, D. (1992). Zero-inflated Poisson regression, with an application to defects in manufacturing. *Technometrics*, 34(1), 1–14. [Link](https://doi.org/10.1080/00401706.1992.10485228), [PDF](http://sci-hub.ren/10.1080/00401706.1992.10485228), [Google](https://scholar.google.com/scholar?q=Zero-inflated+Poisson+regression+with+an+application+to+defects+in+manufacturing).
- Papke, L. E., & Wooldridge, J. M. (1996). Econometric methods for fractional response variables with an application to 401(k) plan participation rates. *Journal of Applied Econometrics*, 11(6), 619–632. [Link](https://doi.org/10.1002/(SICI)1099-1255(199611)11:6%3C619::AID-JAE418%3E3.0.CO;2-1), [PDF](http://sci-hub.ren/10.1002/(SICI)1099-1255(199611)11:6%3C619::AID-JAE418%3E3.0.CO;2-1), [Google](https://scholar.google.com/scholar?q=Econometric+methods+for+fractional+response+variables+with+an+application+to+401(k)+plan+participation+rates).